In [52]:
import json
from typing import Optional

In [53]:
from dotenv import load_dotenv
import os


load_dotenv()

OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
OPEN_ROUTER_COMPLETION_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
OPEN_ROUTER_URL = "https://openrouter.ai/api/v1"

OLLAMA_COMPLETION_MODEL = "qwen3:latest"


OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
OLLAMA_API_KEY='ollama'


OPEN_ROUTER_HEADERS = {
        "HTTP-Referer": "https://github.com/pixelbuildlab/be-story-teller",
        "X-OpenRouter-Title": "Be story teller - Agentic way to stories",
    }

In [54]:
USE_OLLAMA=True
MODEL= OLLAMA_COMPLETION_MODEL if USE_OLLAMA else OPEN_ROUTER_COMPLETION_MODEL
API_URL= OLLAMA_API_URL if USE_OLLAMA else OPEN_ROUTER_URL
API_KEY = OLLAMA_API_KEY if USE_OLLAMA else OPEN_ROUTER_API_KEY
HEADERS = None if USE_OLLAMA else OPEN_ROUTER_HEADERS

In [55]:
USE_OLLAMA, MODEL, API_URL,  HEADERS

(True, 'qwen3:latest', 'http://127.0.0.1:11434/v1', None)

In [56]:
import openai
from openai import OpenAI

client = OpenAI(
    base_url=API_URL,
    api_key=API_KEY,
    default_headers=HEADERS,
)


In [57]:
SYSTEM_META_PROMPT = """
You are a professional children's story writer.

Your goals:
- Write bedtime stories for children aged 4–9.
- Stories should be calming.
- Never include violence or horror.
- Keep language simple.
- If you need to use a tool, use it.
- Stories should be relaxing, pleasing to hear and lesson full.
- Note: Story should be only of 20 characters for now.
"""

In [58]:
def AI(messages: list, tools: Optional[list] | None):
    chat_completion = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )
    return chat_completion

In [59]:
META_PROMPT_OPTIMIZER = """
You are an expert prompt optimizer for children's story generation.

Your task is to transform short or incomplete user requests into rich, detailed prompts for a story-writing AI.

Rules:
- Preserve the user's original intent.
- Add reasonable assumptions when details are missing.
- Specify:
  - protagonist
  - setting
  - conflict
  - tone
  - target age
  - ending
  - approximate length
- Do not write the story.
- Return ONLY the optimized prompt.
"""


async def StoryPromptOptimizer(prompt: str):
    print("starting StoryPromptOptimizer")
    messages = [
        {"role": "system", "content": META_PROMPT_OPTIMIZER},
        {
            "role": "user",
            "content": f"{prompt}",
        },
    ]

    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message

    print("StoryPromptOptimizer called")
    return response_message, chat_completion

In [60]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "StoryPromptOptimizer",
            "description": "Optimize user input prompt to a level it creates stunning storyline",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": "User input prompt to optimize",
                    }
                },
                "required": ["prompt"],
            },
        },
    }
]

In [61]:
MESSAGES_LIST = []

In [62]:
async def openai_chat_completion(prompt: str, META_PROMPT: str):
    try:
        messages = [
            {"role": "system", "content": META_PROMPT},
            {
                "role": "user",
                "content": f"{prompt}",
            },
        ]
        MESSAGES_LIST.extend(messages)

        while True:
            print("STARTING AGENT")
            chat_completion = AI(MESSAGES_LIST, tools)
            response_message = chat_completion.choices[0].message

            MESSAGES_LIST.append(response_message.model_dump())
            print(f"MAIN chat output: {response_message}")

            # If LLM returned tool calls, process them
            if hasattr(response_message, "tool_calls") and response_message.tool_calls:
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    print(f"Tool call: {function_name}, args: {function_args}")
                    agent_args = []

                    tool_function = globals()[function_name]
                    
                    tool_result, tool_chat_completion = await tool_function(
                        *agent_args, **function_args
                    )

                    MESSAGES_LIST.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": tool_result.content,
                        }
                    )

            else:
                # LIKELY TO STOP LOOP
                # IF BUGGY USE A TOOL TO SOP THE LOOP.
                return chat_completion

            # return chat_completion

    except openai.APIConnectionError as e:
        print(f"Network connectivity issue: {e}")
    except openai.RateLimitError as e:
        print(f"Rate limits hit or out of funds: {e}")
    except openai.APIStatusError as e:
        print(f"HTTP Error received (Status: {e.status_code}): {e.response}")

In [63]:
await openai_chat_completion('rabbit', SYSTEM_META_PROMPT)

STARTING AGENT
MAIN chat output: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_bxxts1dn', function=Function(arguments='{"prompt":"rabbit"}', name='StoryPromptOptimizer'), type='function', index=0)], reasoning="Okay, the user wants a bedtime story for kids aged 4-9. The main character is a rabbit. Let me think about how to make this calming and safe.\n\nFirst, I need to set a peaceful scene. Maybe a meadow at sunset? That's soothing. The rabbit should be gentle, perhaps named something like Thistle. Adding elements like stars and a moon can enhance the calming vibe.\n\nI should include other friendly animals. A deer and a fox could work, but they need to be non-threatening. Maybe they join the rabbit for a journey. The story should have a gentle conflict, like a storm, but resolve it safely. The storm can be a challenge they overcome together, teaching resilie

ChatCompletion(id='chatcmpl-418', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**Thimble and the Moonlit Garden**  \n\nThimble the rabbit hopped through the forest, ears twitching at the hum of fireflies. “Where is the garden?” he wondered, nibbling a clover. The stars blinked above, guiding him through whispering trees.  \n\nA thorny bush blocked his path. “I’ll find another way,” Thimble sighed, but the thorns glowed faintly, forming a bridge of light. He tiptoed across, petals brushing his nose.  \n\nAt the garden’s edge, a trickster fox grinned. “Turn back, or I’ll nibble your ears!” Thimble blinked. “You’d only harm yourself,” he said, recalling his grandmother’s tales. The fox snorted, darting away.  \n\nThe garden bloomed with silver flowers and trees that hummed lullabies. A glowing flower floated to Thimble, its light warm as a hug. “Take this,” it whispered, “to heal your friend’s sadness.”  \n\nThimble hurried home, the

In [64]:
MESSAGES_LIST

[{'role': 'system',
  'content': "\nYou are a professional children's story writer.\n\nYour goals:\n- Write bedtime stories for children aged 4–9.\n- Stories should be calming.\n- Never include violence or horror.\n- Keep language simple.\n- If you need to use a tool, use it.\n- Stories should be relaxing, pleasing to hear and lesson full.\n- Note: Story should be only of 20 characters for now.\n"},
 {'role': 'user', 'content': 'rabbit'},
 {'content': '',
  'refusal': None,
  'role': 'assistant',
  'annotations': None,
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_bxxts1dn',
    'function': {'arguments': '{"prompt":"rabbit"}',
     'name': 'StoryPromptOptimizer'},
    'type': 'function',
    'index': 0}],
  'reasoning': "Okay, the user wants a bedtime story for kids aged 4-9. The main character is a rabbit. Let me think about how to make this calming and safe.\n\nFirst, I need to set a peaceful scene. Maybe a meadow at sunset? That's soothing. The rabbit shoul

In [65]:
MESSAGES_LIST[-1]['content']

'**Thimble and the Moonlit Garden**  \n\nThimble the rabbit hopped through the forest, ears twitching at the hum of fireflies. “Where is the garden?” he wondered, nibbling a clover. The stars blinked above, guiding him through whispering trees.  \n\nA thorny bush blocked his path. “I’ll find another way,” Thimble sighed, but the thorns glowed faintly, forming a bridge of light. He tiptoed across, petals brushing his nose.  \n\nAt the garden’s edge, a trickster fox grinned. “Turn back, or I’ll nibble your ears!” Thimble blinked. “You’d only harm yourself,” he said, recalling his grandmother’s tales. The fox snorted, darting away.  \n\nThe garden bloomed with silver flowers and trees that hummed lullabies. A glowing flower floated to Thimble, its light warm as a hug. “Take this,” it whispered, “to heal your friend’s sadness.”  \n\nThimble hurried home, the flower glowing in his paw. He placed it by his sick friend’s nest. The flower’s light wrapped around them, melting worries like snow.